# The Chocolate Study

### What predicts the expert rating of a chocolate bar: cocoa percentage, maker, or bean origin? And which factor matters most?

---

**Course:** Data Science (Module 376)  
**Institution:** HES-SO Valais — Technologies du Vivant  
**Academic Year:** 2025–2026  
**Project:** 5 — Comparaison de barres chocolatées  
**Authors:** Samuele · Lydia · Noah

---

## 1. Introduction

Every chocolate bar carries a story written in three hands: the farmer who grew the bean, the maker who roasted and conched it, and the buyer who chose the percentage. In this study, we ask the data which of these three hands leaves the deepest mark on quality.

We analyze a dataset of expert ratings for over 1,700 chocolate bars and investigate whether the rating can be predicted from three candidate factors: **cocoa percentage**, **bean origin** (terroir), and **maker** (company location).

### Research Question

> What predicts the expert rating of a chocolate bar: cocoa percentage, maker, or bean origin? And which factor matters most?

### Sub-questions

1. **Cocoa %** — Is there a monotonic relationship between cocoa percentage and rating, or does a sweet spot exist?
2. **Bean origin (terroir)** — Are there bean-origin countries with systematically higher ratings? How large is the effect?
3. **Maker / company location** — Is there a "production school" effect (e.g. France vs. Belgium vs. USA)?
4. **Interaction & comparison** — When all factors are combined in a single model, which variable explains the most variance?

### Hypotheses

- **H1:** The relationship between cocoa percentage and rating is not linear; there is a sweet spot around 70–75%.
- **H2:** The geographical origin of the bean has a significant but smaller effect than the maker.
- **H3:** The maker (company location) is the strongest predictor: craftsmanship outweighs raw material.

### Work Divisionthere is a sweet spot around 70–

- **Samuele** — Data exploration & preprocessing  
- **Lydia** — Modeling  
- **Noah** — Interpretation & conclusions

Each cell in this notebook is signed by its author.

## 2. Dataset

### Source and Licensing (FAIR compliance)

- **Name:** Flavors of Cacao — Chocolate Bar Ratings  
- **Source:** *[to be completed — original URL / DOI]*  
- **License:** *[to be completed — e.g. CC0, CC-BY, ODbL]*  
- **Retrieved on:** *[date]*

### Variable Dictionary

| Column | Type | Unit | Description |
|--------|------|------|-------------|
| `company` | categorical | — | Name of the chocolate maker |
| `company_location` | categorical | — | Country where the maker is based |
| `bean_origin` | categorical | — | Country of origin of the cocoa beans |
| `cocoa_percent` | numerical | % | Cocoa solids percentage |
| `rating` | numerical | 1–5 | Expert rating |
| `review_date` | temporal | year | Year of the review |
| `bean_type` | categorical | — | Bean variety (Criollo, Forastero, Trinitario, blend…) |

*Note: column names and types to be verified against the actual dataset.*

## 3. Setup & Imports

*Author: Samuele*

In [ ]:
# Author: Samuele
# ── Data manipulation ──────────────────────────────────────────────────────────
import numpy as np                    # Import NumPy for numerical operations
import pandas as pd                   # Import Pandas for data manipulation and analysis

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt         # Import Matplotlib for creating plots
import matplotlib.ticker as mticker     # Import ticker utilities for axis formatting
import seaborn as sns                   # Import Seaborn for statistical visualizations

# ── Statistics ─────────────────────────────────────────────────────────────────
from scipy import stats                  # Import scipy.stats for statistical tests
import statsmodels.api as sm             # Import Statsmodels for linear modeling
import statsmodels.formula.api as smf    # Import formula API for R-style model syntax

# ── Machine learning ───────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, KFold, cross_val_score  # Import data splitting and cross-validation tools
from sklearn.preprocessing import (
    StandardScaler, PolynomialFeatures, OneHotEncoder, LabelEncoder          # Import preprocessing transformers
)
from sklearn.pipeline import Pipeline                    # Import Pipeline to chain preprocessing and model steps
from sklearn.linear_model import LinearRegression        # Import linear regression model
from sklearn.ensemble import RandomForestRegressor       # Import random forest model
from sklearn.inspection import permutation_importance    # Import permutation importance for feature analysis
from sklearn.metrics import mean_squared_error, r2_score # Import evaluation metrics

# ── Display settings ───────────────────────────────────────────────────────────
pd.set_option("display.max_columns", 30)                    # Show up to 30 columns in DataFrame display
pd.set_option("display.float_format", "{:.3f}".format)      # Format floats to 3 decimal places

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)  # Set Seaborn theme with white grid and muted colors
plt.rcParams["figure.dpi"] = 120                            # Set figure resolution to 120 DPI

print("✅  All libraries imported successfully.")             # Confirm all imports were successful


## 4. Data Loading

*Author: Samuele*

In [ ]:
# Author: Samuele

df = pd.read_csv("chocolate.csv", sep=";")   # Load the CSV file using semicolon as delimiter
print("First 5 rows of the dataset:")         # Print header for the preview
display(df.head())                            # Display the first 5 rows of the dataframe
print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")  # Print number of rows and columns


In [ ]:
# Author: Samuele

print("=== Column Information ===")           # Print section header
df.info()                                     # Display column names, types, and non-null counts
print("\n=== Summary Statistics (numerical columns) ===")  # Print header for statistics section
display(df.describe())                        # Show descriptive statistics for all numerical columns


## 5. Data Cleaning & Preprocessing

*Author: Samuele*

Steps performed in this section:
- Inspect missing values and decide on imputation strategy
- Normalize column names
- Convert cocoa percentage to numeric (strip `%` if needed)
- Consolidate categorical levels (e.g. group rare bean origins)
- Handle duplicates and inconsistent entries

In [ ]:
# Author: Samuele

missing = df.isnull().sum()                   # Count missing values per column
missing_pct = (missing / len(df)) * 100       # Calculate missing percentage for each column
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})  # Build a summary dataframe
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)  # Keep only columns with missing values, sorted by percentage
print("=== Missing Values per Column ===")    # Print section header
display(missing_df)                           # Display the missing values summary table


In [ ]:
# Author: Samuele

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")  # Normalize column names: strip whitespace, lowercase, replace spaces with underscores
df = df.rename(columns={"country_of_bean_origin": "bean_origin"})      # Rename for a shorter and cleaner column name
print("Cleaned column names:")               # Print header for result
print(list(df.columns))                      # Display the updated list of column names


In [ ]:
# Author: Samuele

df["cocoa_percent"] = (
    df["cocoa_percent"]
    .astype(str)                          # Convert values to string to allow string operations
    .str.replace("%", "", regex=False)    # Remove the percentage symbol if present
    .str.strip()                          # Strip any leading or trailing whitespace
    .astype(float)                        # Convert back to float for numerical analysis
)
print(f"Cocoa percent range: {df['cocoa_percent'].min():.1f}% – {df['cocoa_percent'].max():.1f}%")  # Print the value range after conversion


In [ ]:
# Author: Samuele

df["cocoa_percent"] = df["cocoa_percent"].fillna(df["cocoa_percent"].median())  # Fill missing cocoa % values with the median
df["rating"]        = df["rating"].fillna(df["rating"].median())                # Fill missing ratings with the median
for col in ["bean_origin", "company_location", "company"]:                      # Iterate over categorical columns with potential missing values
    df[col] = df[col].fillna("Unknown")                                         # Replace missing categorical values with "Unknown"
print("Missing values after cleaning (key columns):")                           # Print section header
print(df[["cocoa_percent", "rating", "bean_origin", "company_location"]].isnull().sum())  # Verify no missing values remain in key columns


In [ ]:
# Author: Samuele

MIN_SAMPLES = 10                                                        # Minimum number of samples required to keep a category as-is
origin_counts = df["bean_origin"].value_counts()                        # Count occurrences of each bean origin
rare_origins  = origin_counts[origin_counts < MIN_SAMPLES].index        # Identify bean origins with fewer than 10 bars
df["bean_origin"] = df["bean_origin"].replace(rare_origins, "Other")   # Replace rare bean origins with "Other" to reduce noise

loc_counts = df["company_location"].value_counts()                      # Count occurrences of each company location
rare_locs  = loc_counts[loc_counts < MIN_SAMPLES].index                 # Identify company locations with fewer than 10 bars
df["company_location"] = df["company_location"].replace(rare_locs, "Other")  # Replace rare company locations with "Other"

print(f"Unique bean origins after consolidation    : {df['bean_origin'].nunique()}")       # Print number of remaining bean origin categories
print(f"Unique company locations after consolidation: {df['company_location'].nunique()}")  # Print number of remaining company location categories


## 6. Exploratory Data Analysis

*Author: Samuele*

Visual and statistical exploration of the dataset to understand distributions, relationships, and potential outliers before modeling.

### 6.1 Distribution of Ratings

In [ ]:
# Author: Samuele

fig, axes = plt.subplots(1, 2, figsize=(12, 4))   # Create a figure with two side-by-side subplots

axes[0].hist(df["rating"].dropna(), bins=20, color="#8B4513", edgecolor="white")  # Plot histogram of ratings, dropping NaN values
axes[0].set_title("Distribution of Expert Ratings")   # Set title for the histogram
axes[0].set_xlabel("Rating (1–5)")                    # Label the x-axis
axes[0].set_ylabel("Number of Chocolate Bars")        # Label the y-axis

axes[1].boxplot(df["rating"].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor="#D2691E", color="black"))  # Plot a filled boxplot of ratings
axes[1].set_title("Boxplot of Ratings")               # Set title for the boxplot
axes[1].set_ylabel("Rating (1–5)")                    # Label the y-axis

plt.tight_layout()   # Adjust layout to prevent overlap between subplots
plt.show()           # Render and display the figure

print("\n=== Rating Summary Statistics ===")  # Print section header
print(df["rating"].describe())               # Display descriptive statistics for the rating column


### 6.2 Distribution of Cocoa Percentage

In [ ]:
# Author: Samuele

fig, ax = plt.subplots(figsize=(8, 4))   # Create a single plot with specified dimensions
ax.hist(df["cocoa_percent"].dropna(), bins=25, color="#5C3317", edgecolor="white")  # Plot histogram of cocoa percentages, dropping NaN values
ax.axvline(70, color="orange", linestyle="--", linewidth=1.8, label="70% (sweet-spot lower bound)")  # Add vertical line at 70% to mark lower sweet-spot bound
ax.axvline(75, color="gold",   linestyle="--", linewidth=1.8, label="75% (sweet-spot upper bound)")  # Add vertical line at 75% to mark upper sweet-spot bound
ax.set_title("Distribution of Cocoa Percentage")   # Set plot title
ax.set_xlabel("Cocoa Percentage (%)")              # Label the x-axis
ax.set_ylabel("Number of Bars")                    # Label the y-axis
ax.legend()                                        # Display legend for the reference lines
plt.tight_layout()                                 # Adjust layout spacing
plt.show()                                         # Render and display the figure
print("\n=== Cocoa Percentage Summary Statistics ===")   # Print section header
print(df["cocoa_percent"].describe())                    # Display descriptive statistics for cocoa percentage


### 6.3 Rating vs. Cocoa Percentage

In [ ]:
# Group cocoa % into 5 readable ranges and show the mean rating for each group.
# This is easier to read than a scatter plot: one bar per range, clear labels.

df["cocoa_range"] = pd.cut(
    df["cocoa_percent"],
    bins=[0, 65, 70, 75, 80, 100],                          # Define bin edges for the cocoa percentage ranges
    labels=["<65%", "65–70%", "70–75%", "75–80%", ">80%"]  # Assign human-readable labels to each range
)

mean_by_range = df.groupby("cocoa_range", observed=True)["rating"].mean()    # Compute mean rating for each cocoa range
count_by_range = df.groupby("cocoa_range", observed=True)["rating"].count()  # Count number of bars in each cocoa range

fig, ax = plt.subplots(figsize=(8, 4))   # Create a figure for the bar chart

colours = ["#D2B48C", "#A0522D", "#8B4513", "#6B3A2A", "#4A2512"]           # Define a gradient of brown colors, one per range
bars = ax.bar(mean_by_range.index, mean_by_range.values,
              color=colours, edgecolor="white", width=0.6)   # Plot bars with mean ratings per cocoa range

overall = df["rating"].mean()                               # Calculate the global mean rating across all bars
ax.axhline(overall, color="orange", linestyle="--", linewidth=1.8,
           label=f"Overall mean ({overall:.2f})")           # Draw a dashed horizontal line at the overall mean

for bar, count in zip(bars, count_by_range):                                # Loop over each bar and its sample count
    ax.text(bar.get_x() + bar.get_width() / 2,                             # Center text horizontally over the bar
            bar.get_height() + 0.003,                                       # Position text just above the bar
            f"{bar.get_height():.2f}\n(n={count})",                        # Show mean value and sample size
            ha="center", va="bottom", fontsize=8.5)                        # Center-align with small font

ax.set_title("H1 — Mean Rating by Cocoa % Range", fontsize=13)   # Set chart title referencing hypothesis H1
ax.set_xlabel("Cocoa Percentage Range")                           # Label the x-axis
ax.set_ylabel("Mean Expert Rating")                               # Label the y-axis
ax.set_ylim(3.0, 3.55)                                            # Set y-axis limits for better readability
ax.legend()                                                       # Display legend with overall mean reference
plt.tight_layout()                                                # Adjust layout to prevent overlap
plt.show()                                                        # Render and display the chart


### 6.4 Rating by Bean Origin

In [ ]:
# Simple horizontal bar chart: one bar per bean-origin country, length = mean rating.
# Much easier to read than a boxplot when you just want to compare averages.

TOP_N = 10   # Number of top bean origin countries to display

top_origins = df["bean_origin"].value_counts().head(TOP_N).index   # Get the TOP_N most frequent bean origins
mean_by_origin = (
    df[df["bean_origin"].isin(top_origins)]        # Filter dataset to include only the top origins
    .groupby("bean_origin")["rating"]              # Group by bean origin
    .mean()                                        # Compute mean rating per origin
    .sort_values(ascending=True)                   # Sort from lowest to highest for horizontal bar readability
)

overall = df["rating"].mean()   # Calculate overall mean rating for the reference line

fig, ax = plt.subplots(figsize=(9, 5))   # Create figure with specified size

bars = ax.barh(mean_by_origin.index, mean_by_origin.values,
               color="#8B4513", edgecolor="white", height=0.6)   # Draw horizontal bars for each bean origin

ax.axvline(overall, color="orange", linestyle="--", linewidth=1.8,
           label=f"Overall mean ({overall:.2f})")   # Draw dashed line at overall mean for comparison

for bar in bars:                                             # Loop over each horizontal bar
    ax.text(bar.get_width() + 0.003,                        # Position label just to the right of the bar
            bar.get_y() + bar.get_height() / 2,             # Vertically center the label on the bar
            f"{bar.get_width():.2f}",                       # Show the mean rating value
            ha="left", va="center", fontsize=9)             # Left-align text with small font

ax.set_title(f"H2 — Mean Rating by Bean Origin  (Top {TOP_N} Countries)", fontsize=13)   # Set chart title referencing hypothesis H2
ax.set_xlabel("Mean Expert Rating")   # Label the x-axis
ax.set_xlim(3.0, 3.45)               # Set x-axis limits for better readability
ax.legend()                          # Display legend with overall mean reference
plt.tight_layout()                   # Adjust layout spacing
plt.show()                           # Render and display the chart


### 6.5 Rating by Company Location

In [ ]:
# Same simple bar chart as section 6.4, now for the maker's country (H3).

TOP_N = 10   # Number of top company location countries to display

top_locs = df["company_location"].value_counts().head(TOP_N).index   # Get the TOP_N most frequent maker locations
mean_by_loc = (
    df[df["company_location"].isin(top_locs)]       # Filter dataset to include only the top maker locations
    .groupby("company_location")["rating"]          # Group by company location
    .mean()                                         # Compute mean rating per location
    .sort_values(ascending=True)                    # Sort from lowest to highest for horizontal bar readability
)

overall = df["rating"].mean()   # Calculate overall mean rating for the reference line

fig, ax = plt.subplots(figsize=(9, 5))   # Create figure with specified size

bars = ax.barh(mean_by_loc.index, mean_by_loc.values,
               color="#5C3317", edgecolor="white", height=0.6)   # Draw horizontal bars for each company location

ax.axvline(overall, color="orange", linestyle="--", linewidth=1.8,
           label=f"Overall mean ({overall:.2f})")   # Draw dashed line at overall mean for comparison

for bar in bars:                                             # Loop over each horizontal bar
    ax.text(bar.get_width() + 0.003,                        # Position label just to the right of the bar
            bar.get_y() + bar.get_height() / 2,             # Vertically center the label on the bar
            f"{bar.get_width():.2f}",                       # Show the mean rating value
            ha="left", va="center", fontsize=9)             # Left-align text with small font

ax.set_title(f"H3 — Mean Rating by Company Location  (Top {TOP_N} Maker Countries)", fontsize=13)   # Set chart title referencing hypothesis H3
ax.set_xlabel("Mean Expert Rating")   # Label the x-axis
ax.set_xlim(3.0, 3.45)               # Set x-axis limits for better readability
ax.legend()                          # Display legend with overall mean reference
plt.tight_layout()                   # Adjust layout spacing
plt.show()                           # Render and display the chart


### 6.6 Correlation Overview

In [ ]:
# Author: Samuele

numeric_cols = df.select_dtypes(include="number").columns.tolist()  # Select only numeric columns for correlation analysis
corr_matrix  = df[numeric_cols].corr()                              # Compute pairwise Pearson correlation matrix

fig, ax = plt.subplots(figsize=(9, 7))                              # Create a figure for the heatmap
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax)                        # Draw annotated heatmap with diverging color scale centered at 0
ax.set_title("Correlation Matrix — Numerical Variables")            # Set heatmap title
plt.tight_layout()                                                  # Adjust layout spacing
plt.show()                                                          # Render and display the heatmap


## 7. Feature Engineering & Train–Test Split

*Author: Samuele*

In [ ]:
# Author: Samuele

origin_mean = df.groupby("bean_origin")["rating"].mean()           # Compute mean rating for each bean origin
df["bean_origin_enc"] = df["bean_origin"].map(origin_mean)         # Map each bar's bean origin to its group mean rating (target encoding)

loc_mean = df.groupby("company_location")["rating"].mean()         # Compute mean rating for each company location
df["company_location_enc"] = df["company_location"].map(loc_mean)  # Map each bar's company location to its group mean rating (target encoding)

print("Target-encoded columns added successfully.")                 # Confirm the encoding was applied
print("\nSample of the encoding result:")                           # Print header for preview
display(df[["bean_origin", "bean_origin_enc",
            "company_location", "company_location_enc"]].head(8))  # Display a sample to verify the encoding values


In [ ]:
# Author: Samuele

FEATURES = ["cocoa_percent", "bean_origin_enc", "company_location_enc"]  # Define the list of input features for the model
TARGET   = "rating"                                                        # Define the target variable to predict

df_model = df[FEATURES + [TARGET]].dropna()   # Select only relevant columns and drop rows with any missing value
X = df_model[FEATURES]                        # Extract the feature matrix
y = df_model[TARGET]                          # Extract the target vector

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)     # Split into 80% training and 20% test set with a fixed random seed for reproducibility

print(f"Training set : {X_train.shape[0]} bars  ({X_train.shape[0]/len(X)*100:.0f}%)")   # Print training set size
print(f"Test set     : {X_test.shape[0]}  bars  ({X_test.shape[0]/len(X)*100:.0f}%)")    # Print test set size


In [ ]:
# Author: Samuele

scaler = StandardScaler()                           # Instantiate StandardScaler to normalize features to zero mean and unit variance
X_train_scaled = scaler.fit_transform(X_train)      # Fit the scaler on training data and transform it
X_test_scaled  = scaler.transform(X_test)           # Apply the same transformation to test data (no re-fitting to prevent data leakage)
print("✅  Features standardised.")                  # Confirm scaling was applied
print("   Each column now has approximately: mean ≈ 0 and standard deviation ≈ 1 (on training set).")  # Explain the effect of standardization


## 8. Modeling

*Author: Lydia*

We train and evaluate a sequence of models, from the simplest to the most expressive, to test each hypothesis and to identify which factor explains the most variance in expert ratings.

### 8.1 Baseline — Mean Predictor

In [ ]:
# Author: Lydia

mean_rating = y_train.mean()                            # Calcule la note moyenne sur l'ensemble d'entraînement
y_pred_baseline = np.full(len(y_test), mean_rating)     # Crée un tableau de prédictions qui répète toujours la note moyenne

mse_baseline = mean_squared_error(y_test, y_pred_baseline)  # Calcule l'erreur quadratique moyenne du modèle de référence
r2_baseline  = r2_score(y_test, y_pred_baseline)            # Calcule le coefficient de détermination R² du modèle de référence

print("=== Baseline — Mean Predictor ===")                  # Affiche le titre de la section
print(f"  Predicted value (always) : {mean_rating:.4f}")    # Affiche la valeur prédite constante
print(f"  MSE                      : {mse_baseline:.4f}")   # Affiche l'erreur quadratique moyenne
print(f"  RMSE                     : {np.sqrt(mse_baseline):.4f}")  # Affiche la racine de l'erreur quadratique moyenne
print(f"  R²                       : {r2_baseline:.4f}  ← always 0 by definition; this is our floor")  # Affiche le R² (toujours 0 pour un prédicteur constant)


### 8.2 Linear Regression on Cocoa Percentage (H1)

Tests whether cocoa percentage alone is a useful linear predictor of the rating.

In [ ]:
# Author: Lydia

lr_cocoa = LinearRegression()                                  # Instancie un modèle de régression linéaire simple
lr_cocoa.fit(X_train[["cocoa_percent"]], y_train)              # Entraîne le modèle sur le pourcentage de cacao uniquement
y_pred_lr_cocoa = lr_cocoa.predict(X_test[["cocoa_percent"]])  # Prédit les notes sur l'ensemble de test

mse_lr_cocoa = mean_squared_error(y_test, y_pred_lr_cocoa)  # Calcule l'erreur quadratique moyenne
r2_lr_cocoa  = r2_score(y_test, y_pred_lr_cocoa)            # Calcule le coefficient de détermination R²

print("=== Linear Regression — Cocoa % only (H1 linear baseline) ===")  # Affiche le titre de la section
print(f"  Slope (coefficient) : {lr_cocoa.coef_[0]:+.4f}")  # Affiche le coefficient directeur (pente) de la droite de régression
print(f"  Intercept           : {lr_cocoa.intercept_:.4f}")  # Affiche l'ordonnée à l'origine
print(f"  MSE                 : {mse_lr_cocoa:.4f}")         # Affiche l'erreur quadratique moyenne
print(f"  RMSE                : {np.sqrt(mse_lr_cocoa):.4f}")  # Affiche la racine de l'erreur quadratique moyenne
print(f"  R²                  : {r2_lr_cocoa:.4f}")          # Affiche le coefficient de détermination R²


### 8.3 Polynomial Regression on Cocoa Percentage (H1)

Tests the non-linear (sweet-spot) hypothesis.

In [ ]:
# Author: Lydia

# POLYNOMIAL REGRESSION — testing the sweet-spot hypothesis (H1).
# We compare three models on cocoa % alone:
#   • Linear (degree 1)  — straight line
#   • Polynomial degree 2 — one curve (can capture one peak)
#   • Polynomial degree 3 — two curves
# A higher R² means the model explains more of the variation in ratings.

results_poly = {}   # Dictionnaire pour stocker les métriques de chaque modèle

for degree, label in [(1, "Linear"), (2, "Poly degree 2"), (3, "Poly degree 3")]:  # Boucle sur les trois degrés polynomiaux à comparer
    if degree == 1:                                     # Si le degré est 1, utilise directement une régression linéaire simple
        pipe = LinearRegression()
    else:
        pipe = Pipeline([                               # Sinon, crée un pipeline avec transformation polynomiale puis régression
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),  # Génère les caractéristiques polynomiales jusqu'au degré spécifié
            ("lr",   LinearRegression())                # Applique la régression linéaire sur les nouvelles caractéristiques
        ])
    pipe.fit(X_train[["cocoa_percent"]], y_train)       # Entraîne le pipeline sur les données d'entraînement
    y_pred = pipe.predict(X_test[["cocoa_percent"]])    # Prédit les notes sur l'ensemble de test
    results_poly[label] = {
        "MSE": mean_squared_error(y_test, y_pred),      # Calcule et stocke l'erreur quadratique moyenne
        "R²" : r2_score(y_test, y_pred)                 # Calcule et stocke le coefficient de détermination R²
    }
    print(f"  {label:18s} — R²: {results_poly[label]['R²']:.4f}  |  MSE: {results_poly[label]['MSE']:.4f}")  # Affiche les métriques du modèle courant

# ── Simple bar chart: which model fits better? ────────────────────────────────
# Each bar = one model. Taller bar = higher R² = better at predicting the rating.
fig, ax = plt.subplots(figsize=(7, 4))   # Crée une figure pour le graphique en barres

labels = list(results_poly.keys())                          # Récupère les étiquettes des modèles
r2_vals = [results_poly[l]["R²"] for l in labels]          # Récupère les valeurs R² correspondantes
colours = ["#D2B48C", "#8B4513", "#4A2512"]                 # Définit les couleurs pour chaque barre (du plus clair au plus foncé)

bars = ax.bar(labels, r2_vals, color=colours, edgecolor="white", width=0.5)  # Trace les barres avec les valeurs R²

for bar, val in zip(bars, r2_vals):                         # Boucle sur chaque barre pour ajouter une étiquette de valeur
    ax.text(bar.get_x() + bar.get_width() / 2,             # Centre l'étiquette horizontalement au-dessus de la barre
            bar.get_height() + 0.0003,                     # Positionne l'étiquette légèrement au-dessus de la barre
            f"R² = {val:.4f}",                             # Affiche la valeur R² arrondie à 4 décimales
            ha="center", va="bottom", fontsize=9)          # Aligne le texte au centre avec une petite police

ax.set_title("H1 — Which model fits cocoa % → rating best?\n"
             "Higher R² = better prediction", fontsize=12)  # Définit le titre du graphique avec référence à H1
ax.set_ylabel("R² on test set  (higher is better)")        # Étiquette l'axe des ordonnées
ax.set_ylim(0, max(r2_vals) * 2 + 0.005)                   # Ajuste les limites de l'axe y pour une meilleure lisibilité
plt.tight_layout()                                         # Ajuste la mise en page pour éviter les chevauchements
plt.show()                                                 # Affiche le graphique


### 8.4 Multivariate Linear Regression

All features included: cocoa percentage, bean origin, maker (company location).

In [ ]:
# Author: Lydia

lr_full = LinearRegression()                         # Instancie un modèle de régression linéaire multi-varié
lr_full.fit(X_train_scaled, y_train)                 # Entraîne le modèle sur toutes les caractéristiques standardisées
y_pred_lr_full = lr_full.predict(X_test_scaled)      # Prédit les notes sur l'ensemble de test standardisé

mse_lr_full = mean_squared_error(y_test, y_pred_lr_full)  # Calcule l'erreur quadratique moyenne du modèle complet
r2_lr_full  = r2_score(y_test, y_pred_lr_full)            # Calcule le coefficient de détermination R² du modèle complet

print("=== Multivariate Linear Regression (all features) ===")  # Affiche le titre de la section
print(f"  MSE  : {mse_lr_full:.4f}")                            # Affiche l'erreur quadratique moyenne
print(f"  RMSE : {np.sqrt(mse_lr_full):.4f}")                   # Affiche la racine de l'erreur quadratique moyenne
print(f"  R²   : {r2_lr_full:.4f}")                             # Affiche le coefficient de détermination R²
print("\n  Standardised Coefficients:")                          # Affiche le sous-titre pour les coefficients standardisés
for feat, coef in zip(FEATURES, lr_full.coef_):                 # Boucle sur chaque caractéristique et son coefficient standardisé
    bar = "█" * int(abs(coef) * 40)                             # Crée une barre visuelle proportionnelle à la valeur absolue du coefficient
    print(f"    {feat:30s} : {coef:+.4f}  {bar}")              # Affiche le nom, la valeur et la barre visuelle du coefficient


### 8.5 Random Forest Regressor

Non-linear model to capture interactions and to obtain feature importance estimates.

In [ ]:
# Author: Lydia

rf = RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)  # Instancie une forêt aléatoire avec 200 arbres, profondeur illimitée et parallélisme complet
rf.fit(X_train, y_train)        # Entraîne la forêt aléatoire sur les données d'entraînement
y_pred_rf = rf.predict(X_test)  # Prédit les notes sur l'ensemble de test

mse_rf = mean_squared_error(y_test, y_pred_rf)  # Calcule l'erreur quadratique moyenne de la forêt aléatoire
r2_rf  = r2_score(y_test, y_pred_rf)            # Calcule le coefficient de détermination R² de la forêt aléatoire

print("=== Random Forest Regressor (200 trees) ===")  # Affiche le titre de la section
print(f"  MSE  : {mse_rf:.4f}")                        # Affiche l'erreur quadratique moyenne
print(f"  RMSE : {np.sqrt(mse_rf):.4f}")               # Affiche la racine de l'erreur quadratique moyenne
print(f"  R²   : {r2_rf:.4f}")                         # Affiche le coefficient de détermination R²


### 8.6 Cross-Validation

In [ ]:
# Author: Lydia

kf = KFold(n_splits=5, shuffle=True, random_state=42)  # Crée un validateur croisé à 5 plis avec mélange aléatoire des données
cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),  # Définit le modèle à évaluer (forêt aléatoire avec 200 arbres)
    X, y, cv=kf, scoring="r2"                                             # Évalue le modèle sur l'ensemble des données avec le score R²
)

print("=== 5-Fold Cross-Validation — Random Forest ===")           # Affiche le titre de la section
print(f"  R² per fold : {[f'{s:.4f}' for s in cv_scores]}")        # Affiche le R² obtenu pour chaque pli de validation
print(f"  Mean R²     : {cv_scores.mean():.4f}")                    # Affiche la moyenne des R² sur tous les plis
print(f"  Std  R²     : {cv_scores.std():.4f}")                     # Affiche l'écart-type des R² (mesure de la stabilité du modèle)


### 8.7 Residual Analysis

In [ ]:
# Author: Lydia

residuals = y_test - y_pred_rf   # Calcule les résidus comme la différence entre les valeurs réelles et les valeurs prédites

fig, axes = plt.subplots(1, 2, figsize=(12, 4))   # Crée une figure avec deux sous-graphiques côte à côte

axes[0].scatter(y_pred_rf, residuals, alpha=0.4, color="#8B4513", s=15)  # Trace un nuage de points résidus vs valeurs prédites
axes[0].axhline(0, color="red", linewidth=1.5, linestyle="--", label="Residual = 0")  # Trace une ligne horizontale à zéro comme référence idéale
axes[0].set_title("Residuals vs. Predicted Values")   # Définit le titre du premier sous-graphique
axes[0].set_xlabel("Predicted Rating")                # Étiquette l'axe des abscisses
axes[0].set_ylabel("Residual  (Actual − Predicted)")  # Étiquette l'axe des ordonnées
axes[0].legend(fontsize=8)                            # Affiche la légende avec une petite police

axes[1].hist(residuals, bins=30, color="#D2691E", edgecolor="white")   # Trace un histogramme de la distribution des résidus
axes[1].axvline(0, color="red", linewidth=1.5, linestyle="--")        # Trace une ligne verticale à zéro comme référence
axes[1].set_title("Distribution of Residuals")   # Définit le titre du second sous-graphique
axes[1].set_xlabel("Residual")                   # Étiquette l'axe des abscisses
axes[1].set_ylabel("Frequency")                  # Étiquette l'axe des ordonnées

plt.tight_layout()    # Ajuste la mise en page pour éviter les chevauchements
plt.show()            # Affiche les deux graphiques

stat, p_value = stats.shapiro(residuals.values[:500])  # Applique le test de Shapiro-Wilk sur les 500 premiers résidus pour tester la normalité
print(f"Shapiro-Wilk test: W = {stat:.4f},  p-value = {p_value:.4f}")  # Affiche la statistique W et la p-valeur du test de normalité


## 9. Interpretation

*Author: Noah*

We compare the relative contribution of each factor to the prediction, and discuss what the results imply about our three hypotheses.

### 9.1 Feature Importance

In [ ]:
# Author: Noah

importances = rf.feature_importances_      # Extract impurity-based feature importances from the trained random forest
importance_df = pd.DataFrame({
    "Feature"   : FEATURES,               # Feature names list
    "Importance": importances             # Corresponding importance scores
}).sort_values("Importance", ascending=False)  # Sort features from most to least important

print("=== Impurity-Based Feature Importance (Random Forest) ===")   # Print section header
display(importance_df)   # Display the feature importance table

fig, ax = plt.subplots(figsize=(7, 3.5))   # Create figure for the horizontal bar chart
colours = ["#5C3317", "#8B4513", "#D2691E"]   # Define colors for each bar (darkest for highest importance)
ax.barh(importance_df["Feature"], importance_df["Importance"], color=colours)  # Draw horizontal bars with importance values
ax.set_xlabel("Importance Score")          # Label the x-axis
ax.set_title("Feature Importance — Random Forest")   # Set chart title
ax.invert_yaxis()                          # Invert y-axis so the most important feature appears at the top
plt.tight_layout()                         # Adjust layout spacing
plt.show()                                 # Render and display the chart


In [ ]:
# Author: Noah

perm_imp = permutation_importance(rf, X_test, y_test, n_repeats=30, random_state=42)  # Compute permutation importance by shuffling each feature 30 times on the test set

perm_df = pd.DataFrame({
    "Feature"        : FEATURES,                    # Feature names list
    "Mean Importance": perm_imp.importances_mean,   # Mean decrease in R² when each feature is shuffled
    "Std"            : perm_imp.importances_std     # Standard deviation of importance across the 30 repeats
}).sort_values("Mean Importance", ascending=False)  # Sort features from most to least important

print("=== Permutation Feature Importance (test set) ===")   # Print section header
display(perm_df)   # Display the permutation importance table

fig, ax = plt.subplots(figsize=(7, 3.5))   # Create figure for the horizontal bar chart
ax.barh(perm_df["Feature"], perm_df["Mean Importance"],
        xerr=perm_df["Std"], color=["#5C3317", "#8B4513", "#D2691E"],
        align="center", capsize=4)    # Draw horizontal bars with error bars showing standard deviation across repeats
ax.set_xlabel("Mean decrease in R² when feature is shuffled")   # Label the x-axis
ax.set_title("Permutation Feature Importance — Random Forest")  # Set chart title
ax.invert_yaxis()                      # Invert y-axis so the most important feature appears at the top
plt.tight_layout()                     # Adjust layout spacing
plt.show()                             # Render and display the chart


### 9.2 Standardized Coefficients (Linear Model)

In [ ]:
# Author: Noah

coef_df = pd.DataFrame({
    "Feature"                 : FEATURES,        # Feature names list
    "Standardised Coefficient": lr_full.coef_    # Standardised coefficients from the multivariate linear model
})
coef_df["Abs"] = coef_df["Standardised Coefficient"].abs()          # Compute absolute value to enable sorting by magnitude
coef_df = coef_df.sort_values("Abs", ascending=False).drop(columns="Abs")  # Sort by magnitude and remove the helper column

print("=== Standardised Coefficients — Multivariate Linear Regression ===\n")  # Print section header
display(coef_df)   # Display the coefficients table

fig, ax = plt.subplots(figsize=(7, 3.5))   # Create figure for the horizontal bar chart
bar_colours = ["#5C3317" if c > 0 else "#A9A9A9"
               for c in coef_df["Standardised Coefficient"]]  # Assign brown for positive coefficients, grey for negative
ax.barh(coef_df["Feature"], coef_df["Standardised Coefficient"], color=bar_colours)  # Draw horizontal bars for each coefficient
ax.axvline(0, color="black", linewidth=0.8)  # Draw a vertical reference line at zero to distinguish positive from negative effects
ax.set_title("Standardised Coefficients — Multivariate Linear Regression")  # Set chart title
ax.set_xlabel("Coefficient value (standardised units)")   # Label the x-axis
ax.invert_yaxis()                            # Invert y-axis so the most impactful feature appears at the top
plt.tight_layout()                           # Adjust layout spacing
plt.show()                                   # Render and display the chart


### 9.3 Final Comparison Plot

One figure per hypothesis (H1, H2, H3) — designed for presentation.

In [ ]:
# Author: Noah

# Re-use the bar chart from section 6.3 as the H1 presentation figure.
df["cocoa_range"] = pd.cut(
    df["cocoa_percent"],
    bins=[0, 65, 70, 75, 80, 100],                          # Define bin edges for the cocoa percentage ranges
    labels=["<65%", "65–70%", "70–75%", "75–80%", ">80%"]  # Assign human-readable labels to each range
)
mean_by_range  = df.groupby("cocoa_range", observed=True)["rating"].mean()   # Compute mean rating per cocoa range
count_by_range = df.groupby("cocoa_range", observed=True)["rating"].count()  # Count bars in each cocoa range
overall = df["rating"].mean()   # Calculate overall mean rating for the reference line

fig, ax = plt.subplots(figsize=(8, 4))   # Create figure for the bar chart
colours = ["#D2B48C", "#A0522D", "#8B4513", "#6B3A2A", "#4A2512"]  # Define gradient of brown colors, one per range
bars = ax.bar(mean_by_range.index, mean_by_range.values,
              color=colours, edgecolor="white", width=0.6)  # Draw bars with mean ratings per cocoa range
ax.axhline(overall, color="orange", linestyle="--", linewidth=1.8,
           label=f"Overall mean ({overall:.2f})")           # Draw dashed horizontal line at the overall mean
for bar, count in zip(bars, count_by_range):                # Loop over each bar and its sample count
    ax.text(bar.get_x() + bar.get_width() / 2,             # Center label horizontally over the bar
            bar.get_height() + 0.003,                       # Position label just above the bar top
            f"{bar.get_height():.2f}\n(n={count})",        # Show mean value and sample size
            ha="center", va="bottom", fontsize=8.5)        # Center-align text with small font
ax.set_title("H1 — Mean Rating by Cocoa % Range\nDoes a sweet spot exist around 70–75%?", fontsize=12)  # Set chart title referencing H1
ax.set_xlabel("Cocoa Percentage Range")   # Label the x-axis
ax.set_ylabel("Mean Expert Rating")       # Label the y-axis
ax.set_ylim(3.0, 3.55)                    # Set y-axis limits for better readability
ax.legend()                               # Display legend with overall mean reference
plt.tight_layout()                        # Adjust layout spacing
plt.show()                                # Render and display the chart


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PRESENTATION FIGURE — H2: Does bean origin affect quality?
# Simple bar chart: one bar per country, sorted by mean rating.
# ═══════════════════════════════════════════════════════════════════

TOP_N = 12   # Number of top bean origin countries to display

top_origins = df["bean_origin"].value_counts().head(TOP_N).index   # Get the TOP_N most frequent bean origins
h2_means = (
    df[df["bean_origin"].isin(top_origins)]     # Filter dataset to include only the top bean origins
    .groupby("bean_origin")["rating"]           # Group by bean origin
    .mean()                                     # Compute mean rating per origin
    .sort_values(ascending=True)                # Sort from lowest to highest for horizontal bar readability
)

overall_mean = df["rating"].mean()   # Calculate overall mean rating for the reference line

fig, ax = plt.subplots(figsize=(10, 5))   # Create figure with specified dimensions

colours = ["#c8864e" if v >= overall_mean else "#a0522d" for v in h2_means.values]  # Assign lighter color to origins above average, darker to those below
bars = ax.barh(h2_means.index, h2_means.values,
               color=colours, edgecolor="white", height=0.65)  # Draw horizontal bars for each bean origin

ax.axvline(overall_mean, color="orange", linestyle="--", linewidth=2,
           label=f"Overall mean ({overall_mean:.2f})")   # Draw dashed line at overall mean for visual comparison

for bar in bars:                                         # Loop over each horizontal bar
    ax.text(bar.get_width() + 0.003,                    # Position label just to the right of the bar end
            bar.get_y() + bar.get_height() / 2,         # Vertically center the label on the bar
            f"{bar.get_width():.2f}",                   # Show the mean rating value
            ha="left", va="center", fontsize=9)         # Left-align text with small font

ax.set_title(f"H2 — Mean Rating by Bean Origin  (Top {TOP_N} Countries)\n"
             "Lighter bars = above overall average", fontsize=12)  # Set chart title referencing hypothesis H2
ax.set_xlabel("Mean Expert Rating")   # Label the x-axis
ax.set_xlim(3.0, 3.5)                # Set x-axis limits for better readability
ax.legend()                          # Display legend with overall mean reference
plt.tight_layout()                   # Adjust layout spacing
plt.show()                           # Render and display the chart


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PRESENTATION FIGURE — H3: Is the maker's country the strongest predictor?
# Simple bar chart: one bar per maker country, sorted by mean rating.
# ═══════════════════════════════════════════════════════════════════

TOP_N = 12   # Number of top company location countries to display

top_locs = df["company_location"].value_counts().head(TOP_N).index   # Get the TOP_N most frequent maker locations
h3_means = (
    df[df["company_location"].isin(top_locs)]    # Filter dataset to include only the top maker locations
    .groupby("company_location")["rating"]       # Group by company location
    .mean()                                      # Compute mean rating per location
    .sort_values(ascending=True)                 # Sort from lowest to highest for horizontal bar readability
)

overall_mean = df["rating"].mean()   # Calculate overall mean rating for the reference line

fig, ax = plt.subplots(figsize=(10, 5))   # Create figure with specified dimensions

colours = ["#8fad6e" if v >= overall_mean else "#5c7a3e" for v in h3_means.values]  # Assign lighter color to locations above average, darker to those below
bars = ax.barh(h3_means.index, h3_means.values,
               color=colours, edgecolor="white", height=0.65)  # Draw horizontal bars for each company location

ax.axvline(overall_mean, color="orange", linestyle="--", linewidth=2,
           label=f"Overall mean ({overall_mean:.2f})")   # Draw dashed line at overall mean for visual comparison

for bar in bars:                                         # Loop over each horizontal bar
    ax.text(bar.get_width() + 0.003,                    # Position label just to the right of the bar end
            bar.get_y() + bar.get_height() / 2,         # Vertically center the label on the bar
            f"{bar.get_width():.2f}",                   # Show the mean rating value
            ha="left", va="center", fontsize=9)         # Left-align text with small font

ax.set_title(f"H3 — Mean Rating by Company Location  (Top {TOP_N} Maker Countries)\n"
             "Lighter bars = above overall average", fontsize=12)  # Set chart title referencing hypothesis H3
ax.set_xlabel("Mean Expert Rating")   # Label the x-axis
ax.set_xlim(3.0, 3.5)                # Set x-axis limits for better readability
ax.legend()                          # Display legend with overall mean reference
plt.tight_layout()                   # Adjust layout spacing
plt.show()                           # Render and display the chart

# ── Final Model Performance Summary ──────────────────────────────────────────
print("\n" + "="*60)                              # Print top separator line
print("  MODEL PERFORMANCE SUMMARY (test set)")  # Print section header
print("="*60)                                     # Print separator line

summary = pd.DataFrame({
    "Model": [
        "① Baseline (always predict mean)",          # Label for the baseline model
        "② Linear regression — cocoa % only  (H1)",  # Label for the cocoa-only linear model
        "③ Multivariate linear regression",           # Label for the full multivariate linear model
        "④ Random Forest (200 trees)"                 # Label for the random forest model
    ],
    "R²  (higher is better)": [
        round(r2_baseline,           4),   # R² score for the baseline
        round(r2_lr_cocoa,           4),   # R² score for cocoa-only linear regression
        round(r2_lr_full,            4),   # R² score for full multivariate linear regression
        round(r2_rf,                 4)    # R² score for random forest
    ],
    "RMSE (lower is better)": [
        round(np.sqrt(mse_baseline), 4),   # RMSE for the baseline
        round(np.sqrt(mse_lr_cocoa), 4),   # RMSE for cocoa-only linear regression
        round(np.sqrt(mse_lr_full),  4),   # RMSE for full multivariate linear regression
        round(np.sqrt(mse_rf),       4)    # RMSE for random forest
    ]
})
display(summary)   # Display the model performance comparison table


## 10. Discussion & Conclusions

*Author: Noah*

### Summary of Findings

*[To be completed after analysis. Address each hypothesis:]*

- **H1 — Cocoa percentage and the sweet spot:** *[supported / partially supported / rejected — with evidence]*
- **H2 — Bean origin effect:** *[supported / partially supported / rejected — with evidence]*
- **H3 — Maker as strongest predictor:** *[supported / partially supported / rejected — with evidence]*

### Limitations

- Expert ratings are subjective and may carry reviewer bias.
- The dataset reflects bars that reached the reviewer's attention — selection bias is possible.
- Some bean origin / company location categories have very few samples.
- Causality cannot be inferred from observational data.

### Future Work

- Incorporate sensory descriptors (most memorable characteristics) via NLP.
- Time-series analysis: did ratings shift across reviewing years?
- Compare with consumer-rating datasets (e.g. retail reviews).

### Final Takeaway

*[One paragraph — what would you tell a chocolate maker based on this analysis?]*

## 11. Reproducibility

*Author: Samuele*

FAIR principle R1.2 — provenance. The cell below records the software environment used to produce this notebook.

In [ ]:
# Author: Samuele

import sys, sklearn, scipy, statsmodels   # Import versioning modules for the reproducibility report

print("=" * 55)                                        # Print top separator line
print("  REPRODUCIBILITY — Software Environment")     # Print section title
print("=" * 55)                                        # Print separator line
print(f"  Python version       : {sys.version.split()[0]}")         # Print the Python version number
print(f"  NumPy                : {np.__version__}")                  # Print NumPy version
print(f"  Pandas               : {pd.__version__}")                  # Print Pandas version
print(f"  Matplotlib           : {plt.matplotlib.__version__}")      # Print Matplotlib version
print(f"  Seaborn              : {sns.__version__}")                  # Print Seaborn version
print(f"  Scikit-learn         : {sklearn.__version__}")             # Print Scikit-learn version
print(f"  SciPy                : {scipy.__version__}")               # Print SciPy version
print(f"  Statsmodels          : {statsmodels.__version__}")         # Print Statsmodels version
print("=" * 55)                                                       # Print bottom separator line
print("  Dataset : chocolate.csv  (2,224 bars, semicolon-separated)")  # Print dataset filename and description
print("  Notebook: chocolate_study.ipynb")                            # Print notebook filename


---

*End of notebook — The Chocolate Study, HES-SO Valais 2026.*